In [1]:
import ee
import geemap

# 1. Xác thực và Khởi tạo Google Earth Engine
try:
    ee.Initialize()
    print("Khởi tạo GEE thành công!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Đã xác thực và khởi tạo GEE!")

# 2. Lấy ranh giới hành chính TP. Hồ Chí Minh (ROI) từ bộ dữ liệu FAO
# Lưu ý: Tên chuẩn trong database của FAO là 'Ho Chi Minh City'
hcm_roi = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM1_NAME', 'Ho Chi Minh City'))

# 3. Trực quan hóa thử ranh giới lên bản đồ
Map = geemap.Map()
Map.centerObject(hcm_roi, 10) # Zoom level 10
Map.addLayer(hcm_roi, {'color': 'red'}, 'TP. Ho Chi Minh ROI')
Map 

Khởi tạo GEE thành công!


c:\Users\Tan Nguyen\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


Map(center=[10.752267385049521, 106.69138504342378], controls=(WidgetControl(options=['position', 'transparent…

In [2]:
import ee
import geemap
import warnings

# Tắt các cảnh báo DeprecationWarning cho file code gọn gàng
warnings.filterwarnings('ignore')

# 1. Trở về bộ dữ liệu ranh giới chuẩn ban đầu (chắc chắn chạy được)
hcm_roi = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM1_NAME', 'Ho Chi Minh City'))

# 2. Hàm lọc mây (Cloud Masking) dựa trên dải QA_PIXEL của Landsat 8/9
def maskL89Clouds(image):
    qa = image.select('QA_PIXEL')
    cloudBitMask = 1 << 3
    shadowBitMask = 1 << 4
    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
             .And(qa.bitwiseAnd(shadowBitMask).eq(0))
    return image.updateMask(mask)

# 3. Hàm nội suy hệ số scale (Scale Factors) cho Landsat 8 Level 2
def apply_scale_factors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True).addBands(thermalBands, None, True)

# 4. Truy vấn tập dữ liệu Landsat 8 (Giai đoạn mùa khô 2024 để ít mây nhất)
l8_collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                 .filterBounds(hcm_roi)
                 .filterDate('2024-01-01', '2024-04-30') # Lọc ảnh 4 tháng đầu năm
                 .filter(ee.Filter.lt('CLOUD_COVER', 20)) # Lọc ảnh có độ phủ mây < 20%
                 .map(maskL89Clouds)       # Áp dụng hàm khử mây
                 .map(apply_scale_factors) # Áp dụng hàm scale
                 .median()                 # Lấy trung vị để ghép ra bức ảnh trong trẻo nhất
                 .clip(hcm_roi))           # Cắt đúng theo ranh giới TP.HCM

# 5. Trực quan hóa ảnh vệ tinh thực tế lên bản đồ
Map2 = geemap.Map()
Map2.centerObject(hcm_roi, 10)

# Cấu hình dải màu hiển thị (Băng 4-Đỏ, Băng 3-Xanh lục, Băng 2-Xanh lam)
vis_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0.0,
    'max': 0.3
}

Map2.addLayer(l8_collection, vis_params, 'Landsat 8 - True Color 2024')
Map2.addLayer(hcm_roi, {'color': 'red', 'fillColor': '00000000'}, 'Ranh gioi TP.HCM')
Map2

Map(center=[10.752267385049521, 106.69138504342378], controls=(WidgetControl(options=['position', 'transparent…

In [3]:
# 1. Tính toán Chỉ số thực vật (NDVI)
ndvi = l8_collection.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

# 2. Tính toán Nhiệt độ bề mặt (LST) sang độ C
# Hàm apply_scale_factors trước đó đã chuyển ST_B10 thành nhiệt độ Kelvin
lst = l8_collection.select('ST_B10').subtract(273.15).rename('LST')

# 3. Trực quan hóa LST và NDVI lên bản đồ
Map3 = geemap.Map()
Map3.centerObject(hcm_roi, 10)

# Cấu hình dải màu NDVI (Từ đỏ/vàng: đất trống/đô thị -> Xanh lục: thực vật dày)
ndvi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': ['red', 'yellow', 'lightgreen', 'green', 'darkgreen']
}
Map3.addLayer(ndvi, ndvi_vis, 'Chỉ số NDVI')

# Cấu hình dải màu LST (Từ xanh dương/lục: Mát mẻ -> Vàng/Đỏ: Đảo nhiệt)
lst_vis = {
    'min': 25, 
    'max': 45, # Nhiệt độ giả định từ 25 đến 45 độ C
    'palette': ['blue', 'cyan', 'green', 'yellow', 'red', 'darkred']
}
Map3.addLayer(lst, lst_vis, 'Nhiệt độ LST (°C)')

# Khắc phục lỗi che khuất: Đục rỗng ranh giới, chỉ vẽ viền đỏ độ dày 2 pixel
empty_hcm = ee.Image().byte().paint(hcm_roi, 0, 2)
Map3.addLayer(empty_hcm, {'palette': ['red']}, 'Viền ranh giới TP.HCM')

Map3

Map(center=[10.752267385049521, 106.69138504342378], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
import pandas as pd

# 1. Gom các dải dữ liệu (Bands) cần thiết vào một bức ảnh (Image) duy nhất
training_image = ee.Image.cat([
    lst,
    ndvi,
    ee.Image.pixelLonLat()
])

# 2. Lấy mẫu ngẫu nhiên (Random Sampling) các pixel trong ranh giới TP.HCM
print("Đang trích xuất dữ liệu từ Google Earth Engine, vui lòng đợi khoảng 10-20 giây...")
sample_data = training_image.sample(
    region=hcm_roi.geometry(),
    scale=30,           
    numPixels=5000,     
    seed=42,            
    geometries=False    
)

# 3. Cách ép kiểu an toàn 100% (Không dùng geemap nữa)
# Kéo dữ liệu dạng JSON từ máy chủ Google về và chuyển sang Pandas DataFrame
features = sample_data.getInfo()['features']
data_list = [f['properties'] for f in features]
df = pd.DataFrame(data_list)

# 4. Tiền xử lý: Loại bỏ các dòng chứa giá trị rỗng (NaN) do bị cắt ở viền bản đồ
df = df.dropna()

# 5. Lưu lại thành file CSV để làm điểm checkpoint cho các bước sau
df.to_csv("HCMC_UHI_Dataset_2024.csv", index=False)

# Hiển thị 5 dòng đầu tiên để kiểm tra thành quả
print(f"Trích xuất thành công! Kích thước ma trận dữ liệu: {df.shape}")
display(df.head())

Đang trích xuất dữ liệu từ Google Earth Engine, vui lòng đợi khoảng 10-20 giây...
Trích xuất thành công! Kích thước ma trận dữ liệu: (4980, 4)


,LST,NDVI,latitude,longitude
0,31.378455,0.842087,10.476198,106.833717
1,41.065124,0.213465,10.898496,106.629440
2,37.976943,0.550038,11.013570,106.494424
3,30.501733,0.880452,10.511771,106.817548
4,32.388480,0.535056,10.410711,106.903516


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# 1. Tách biến đầu vào (X) và biến mục tiêu (y)
X = df[['NDVI', 'latitude', 'longitude']]
y = df['LST']

# 2. Chia tập dữ liệu: 80% để huấn luyện (Train), 20% để kiểm thử (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Số lượng mẫu huấn luyện: {X_train.shape[0]}")
print(f"Số lượng mẫu kiểm thử để đánh giá: {X_test.shape[0]}\n")

# 3. Khởi tạo mô hình
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

# 4. Huấn luyện mô hình
print("Đang huấn luyện thuật toán Random Forest...")
rf_model.fit(X_train, y_train)

print("Đang huấn luyện thuật toán XGBoost...")
xgb_model.fit(X_train, y_train)

# 5. Hàm đánh giá mô hình dựa trên tiêu chí đề cương
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"--- Kết quả của {name} ---")
    print(f"RMSE: {rmse:.2f} °C \t(Tiêu chí: <= 2.0°C)")
    print(f"MAE:  {mae:.2f} °C \t(Tiêu chí: <= 1.5°C)")
    print(f"R²:   {r2:.2f} \t\t(Tiêu chí: >= 0.80)\n")
    return r2

# 6. Chạy đánh giá và so sánh
r2_rf = evaluate_model("Random Forest", rf_model, X_test, y_test)
r2_xgb = evaluate_model("XGBoost", xgb_model, X_test, y_test)

# 7. Xuất và lưu trữ mô hình tốt nhất để dùng cho WebGIS
best_model = rf_model if r2_rf > r2_xgb else xgb_model
model_name = "RandomForest" if r2_rf > r2_xgb else "XGBoost"

joblib.dump(best_model, 'best_uhi_model.joblib')
print(f"=> Đã lưu mô hình chiến thắng ({model_name}) vào file 'best_uhi_model.joblib'")

Số lượng mẫu huấn luyện: 3984
Số lượng mẫu kiểm thử để đánh giá: 996

Đang huấn luyện thuật toán Random Forest...
Đang huấn luyện thuật toán XGBoost...
--- Kết quả của Random Forest ---
RMSE: 2.57 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.52 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.68 		(Tiêu chí: >= 0.80)

--- Kết quả của XGBoost ---
RMSE: 2.55 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.50 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.68 		(Tiêu chí: >= 0.80)

=> Đã lưu mô hình chiến thắng (XGBoost) vào file 'best_uhi_model.joblib'


In [ ]:
import pandas as pd

# 1. Tính toán thêm Chỉ số mật độ xây dựng (NDBI)
ndbi = l8_collection.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')

# 2. Gom các dải dữ liệu (Thêm NDBI vào ma trận)
training_image_v2 = ee.Image.cat([
    lst,
    ndvi,
    ndbi,  # Biến số quan trọng quyết định nhiệt độ đô thị
    ee.Image.pixelLonLat()
])

# 3. Lấy mẫu ngẫu nhiên (Giới hạn tối đa 5000 điểm cho hàm getInfo)
print("Đang trích xuất bộ dữ liệu V2 (bổ sung NDBI), vui lòng đợi...")
sample_data_v2 = training_image_v2.sample(
    region=hcm_roi.geometry(),
    scale=30,           
    numPixels=5000,     # Hạ xuống 5000 để vượt qua giới hạn của Google
    seed=42,            
    geometries=False    
)

# 4. Kéo dữ liệu về Pandas DataFrame
features_v2 = sample_data_v2.getInfo()['features']
df_v2 = pd.DataFrame([f['properties'] for f in features_v2])

# Tiền xử lý: Bỏ giá trị rỗng
df_v2 = df_v2.dropna()

# 5. Lưu bộ dữ liệu mới và xem thử
df_v2.to_csv("HCMC_UHI_Dataset_2024_V2.csv", index=False)
print(f"Trích xuất thành công V2! Kích thước ma trận dữ liệu: {df_v2.shape}")
display(df_v2.head())

Đang trích xuất bộ dữ liệu V2 (bổ sung NDBI), vui lòng đợi...
Trích xuất thành công V2! Kích thước ma trận dữ liệu: (4978, 5)


,LST,NDBI,NDVI,latitude,longitude
0,31.378455,-0.585540,0.842087,10.476198,106.833717
1,41.065124,0.066893,0.213465,10.898496,106.629440
2,37.976943,-0.068366,0.550038,11.013570,106.494424
3,30.501733,-0.622916,0.880452,10.511771,106.817548
4,32.388480,-0.386235,0.535056,10.410711,106.903516


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# 1. Tách biến đầu vào (X) có thêm NDBI và biến mục tiêu (y)
X = df_v2[['NDVI', 'NDBI', 'latitude', 'longitude']]
y = df_v2['LST']

# 2. Chia tập dữ liệu: 80% để huấn luyện (Train), 20% để kiểm thử (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Số lượng mẫu huấn luyện: {X_train.shape[0]}")
print(f"Số lượng mẫu kiểm thử để đánh giá: {X_test.shape[0]}\n")

# 3. Khởi tạo mô hình
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

# 4. Huấn luyện mô hình
print("Đang huấn luyện thuật toán Random Forest...")
rf_model.fit(X_train, y_train)

print("Đang huấn luyện thuật toán XGBoost...")
xgb_model.fit(X_train, y_train)

# 5. Hàm đánh giá mô hình
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"--- Kết quả của {name} ---")
    print(f"RMSE: {rmse:.2f} °C \t(Tiêu chí: <= 2.0°C)")
    print(f"MAE:  {mae:.2f} °C \t(Tiêu chí: <= 1.5°C)")
    print(f"R²:   {r2:.2f} \t\t(Tiêu chí: >= 0.80)\n")
    return r2

# 6. Chạy đánh giá và so sánh
r2_rf = evaluate_model("Random Forest", rf_model, X_test, y_test)
r2_xgb = evaluate_model("XGBoost", xgb_model, X_test, y_test)

# 7. Xuất và lưu trữ mô hình tốt nhất
best_model = rf_model if r2_rf > r2_xgb else xgb_model
model_name = "RandomForest" if r2_rf > r2_xgb else "XGBoost"

joblib.dump(best_model, 'best_uhi_model_v2.joblib')
print(f"=> Đã lưu mô hình chiến thắng ({model_name}) vào file 'best_uhi_model_v2.joblib'")

Số lượng mẫu huấn luyện: 3982
Số lượng mẫu kiểm thử để đánh giá: 996

Đang huấn luyện thuật toán Random Forest...
Đang huấn luyện thuật toán XGBoost...
--- Kết quả của Random Forest ---
RMSE: 1.93 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.33 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.79 		(Tiêu chí: >= 0.80)

--- Kết quả của XGBoost ---
RMSE: 1.93 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.31 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.79 		(Tiêu chí: >= 0.80)

=> Đã lưu mô hình chiến thắng (XGBoost) vào file 'best_uhi_model_v2.joblib'


In [ ]:
from sklearn.model_selection import GridSearchCV

print("Đang chạy Grid Search để rà soát siêu tham số tối ưu cho XGBoost...")
print("Quá trình này có thể mất 1-2 phút, vui lòng đợi...\n")

# 1. Khai báo lưới tham số cần thử nghiệm (Tăng số lượng cây, chỉnh độ sâu và tốc độ học)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.8, 1.0]
}

# 2. Khởi tạo mô hình và Grid Search
xgb_base = XGBRegressor(random_state=42)
grid_search = GridSearchCV(
    estimator=xgb_base, 
    param_grid=param_grid, 
    cv=3,             # Chia tập Train làm 3 phần để kiểm định chéo (Cross-validation)
    scoring='r2',     # Mục tiêu tối thượng là tối đa hóa R2
    n_jobs=-1,        # Dùng toàn bộ lõi CPU để chạy cho nhanh
    verbose=1
)

# 3. Tiến hành đào tạo và tìm kiếm
grid_search.fit(X_train, y_train)

# 4. Trích xuất mô hình xịn nhất
best_xgb_tuned = grid_search.best_estimator_
print("\n=> Đã tìm thấy bộ tham số tối ưu:", grid_search.best_params_)
print("-" * 40)

# 5. Đánh giá lại với mô hình đã độ tham số
r2_final = evaluate_model("XGBoost (Đã tinh chỉnh)", best_xgb_tuned, X_test, y_test)

# 6. Lưu file chính thức
if r2_final >= 0.80:
    joblib.dump(best_xgb_tuned, 'final_uhi_model.joblib')
    print("=> Chúc mừng! Đã lưu mô hình hoàn chỉnh vào 'final_uhi_model.joblib'")

Đang chạy Grid Search để rà soát siêu tham số tối ưu cho XGBoost...
Quá trình này có thể mất 1-2 phút, vui lòng đợi...

Fitting 3 folds for each of 54 candidates, totalling 162 fits

=> Đã tìm thấy bộ tham số tối ưu: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
----------------------------------------
--- Kết quả của XGBoost (Đã tinh chỉnh) ---
RMSE: 1.91 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.30 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.80 		(Tiêu chí: >= 0.80)



In [ ]:
import ee
import pandas as pd

# Khởi tạo danh sách rỗng để hứng dữ liệu từng năm
all_dataframes = []

print("Bắt đầu tải dữ liệu trực tiếp về máy tính (không cần qua Google Drive)...")

# Vòng lặp quét qua 10 năm (từ 2017 đến hết 2026)
for year in range(2017, 2027):
    print(f"Đang xử lý và kéo dữ liệu năm {year}...")
    
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)
    
    # Lọc và đục mây ảnh Landsat theo năm
    yearly_col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                 .filterBounds(hcm_roi)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUD_COVER', 20))
                 .map(maskL89Clouds)
                 .map(apply_scale_factors)
                 .median()
                 .clip(hcm_roi))
    
    # Tính LST, NDVI, NDBI
    ndvi = yearly_col.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    ndbi = yearly_col.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')
    lst = yearly_col.select('ST_B10').subtract(273.15).rename('LST')
    
    # Gắn nhãn Năm (Year) vào dữ liệu
    year_img = ee.Image.constant(year).rename('Year').toInt()
    
    combined = ee.Image.cat([lst, ndvi, ndbi, year_img, ee.Image.pixelLonLat()])
    
    # Lấy 2000 điểm (Vượt qua an toàn mức chặn 5000 của getInfo)
    samples = combined.sample(
        region=hcm_roi.geometry(),
        scale=30,
        numPixels=2000,
        seed=42,
        geometries=False
    )
    
    # Ép kiểu trực tiếp về Pandas DataFrame cục bộ
    try:
        features = samples.getInfo()['features']
        df_year = pd.DataFrame([f['properties'] for f in features])
        all_dataframes.append(df_year)
    except Exception as e:
        print(f"Lỗi nhẹ ở năm {year}, bỏ qua và tiếp tục: {e}")

# Gom toàn bộ dữ liệu 10 năm lại thành một bảng duy nhất
final_df = pd.concat(all_dataframes, ignore_index=True)

# Dọn dẹp dữ liệu rỗng (NaN)
final_df = final_df.dropna()

# Lưu thẳng vào ổ cứng máy tính (cùng thư mục với file code)
final_df.to_csv("HCMC_UHI_10Years_2017_2026.csv", index=False)

print(f"\n=> Xong! Đã lưu file 'HCMC_UHI_10Years_2017_2026.csv' trực tiếp vào máy tính.")
print(f"=> Tổng kích thước siêu ma trận: {final_df.shape}")
display(final_df.head(3))
display(final_df.tail(3))

Bắt đầu tải dữ liệu trực tiếp về máy tính (không cần qua Google Drive)...
Đang xử lý và kéo dữ liệu năm 2017...
Đang xử lý và kéo dữ liệu năm 2018...
Đang xử lý và kéo dữ liệu năm 2019...
Đang xử lý và kéo dữ liệu năm 2020...
Đang xử lý và kéo dữ liệu năm 2021...
Đang xử lý và kéo dữ liệu năm 2022...
Đang xử lý và kéo dữ liệu năm 2023...
Đang xử lý và kéo dữ liệu năm 2024...
Đang xử lý và kéo dữ liệu năm 2025...
Đang xử lý và kéo dữ liệu năm 2026...

=> Xong! Đã lưu file 'HCMC_UHI_10Years_2017_2026.csv' trực tiếp vào máy tính.
=> Tổng kích thước siêu ma trận: (19937, 6)


,LST,NDBI,NDVI,Year,latitude,longitude
0,32.784970,-0.580002,0.840443,2017,10.476198,106.833717
1,40.111496,-0.000094,0.247263,2017,10.898496,106.629440
2,33.639475,-0.178130,0.661504,2017,11.013570,106.494424


,LST,NDBI,NDVI,Year,latitude,longitude
19934,30.354758,-0.675336,0.873532,2026,10.447901,106.849617
19935,38.628075,0.037392,0.469981,2026,10.772103,106.489842
19936,44.382312,0.081525,0.114782,2026,10.868851,106.763110


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# 1. Khai báo biến đầu vào (X) - Chú ý ĐÃ CÓ THÊM CỘT 'Year'
X = final_df[['NDVI', 'NDBI', 'Year', 'latitude', 'longitude']]
y = final_df['LST']

# 2. Chia tập dữ liệu: 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước tập huấn luyện: {X_train.shape[0]} mẫu")
print(f"Kích thước tập kiểm thử để đánh giá: {X_test.shape[0]} mẫu\n")

# 3. Khởi tạo thuật toán XGBoost với bộ tham số xịn nhất đã tìm được
xgb_ultimate = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

# 4. Huấn luyện mô hình
print("Đang huấn luyện mô hình XGBoost với dữ liệu lịch sử 10 năm...")
xgb_ultimate.fit(X_train, y_train)

# 5. Đánh giá độ chính xác
y_pred = xgb_ultimate.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- KẾT QUẢ NGHIỆM THU CUỐI CÙNG ---")
print(f"RMSE: {rmse:.2f} °C \t(Tiêu chí: <= 2.0°C)")
print(f"MAE:  {mae:.2f} °C \t(Tiêu chí: <= 1.5°C)")
print(f"R²:   {r2:.2f} \t\t(Tiêu chí: >= 0.80)\n")

# 6. Lưu file model chính thức
joblib.dump(xgb_ultimate, 'ultimate_uhi_model_10yrs.joblib')
print("=> XONG! File 'ultimate_uhi_model_10yrs.joblib' đã sẵn sàng.")
print("=> Bây giờ, chỉ cần truyền vào tham số Year=2030, mô hình có thể dự báo được tương lai!")

Kích thước tập huấn luyện: 15949 mẫu
Kích thước tập kiểm thử để đánh giá: 3988 mẫu

Đang huấn luyện mô hình XGBoost với dữ liệu lịch sử 10 năm...
--- KẾT QUẢ NGHIỆM THU CUỐI CÙNG ---
RMSE: 1.93 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.41 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.84 		(Tiêu chí: >= 0.80)

=> XONG! File 'ultimate_uhi_model_10yrs.joblib' đã sẵn sàng.
=> Bây giờ, chỉ cần truyền vào tham số Year=2030, mô hình có thể dự báo được tương lai!


In [5]:
# Dán vào một cell của notebook (sau cell 8), chạy 1 lần. Dùng lại hcm_roi, maskL89Clouds, apply_scale_factors.
import ee, json, os, requests
os.makedirs('data/overlays', exist_ok=True)
geom = hcm_roi.geometry()
co = geom.bounds().getInfo()['coordinates'][0]
lons, lats = [p[0] for p in co], [p[1] for p in co]
json.dump({'west': min(lons), 'east': max(lons), 'south': min(lats), 'north': max(lats)}, open('data/overlays/bounds.json', 'w'))
json.dump(geom.getInfo(), open('data/hcm_boundary.geojson', 'w'), ensure_ascii=False)   # ranh giới FAO GAUL giống bản demo
PAL = {'LST': (['blue', 'cyan', 'green', 'yellow', 'red'], 25, 45),
       'NDVI': (['red', 'yellow', 'lightgreen', 'green', 'darkgreen'], -0.2, 0.8),
       'NDBI': (['green', 'yellow', 'red'], -0.5, 0.5)}
for year in range(2017, 2027):
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(hcm_roi)
           .filterDate(f'{year}-01-01', f'{year}-12-31').filter(ee.Filter.lt('CLOUD_COVER', 20))
           .map(maskL89Clouds).map(apply_scale_factors).median().clip(hcm_roi))
    bands = {'LST': col.select('ST_B10').subtract(273.15),
             'NDVI': col.normalizedDifference(['SR_B5', 'SR_B4']),
             'NDBI': col.normalizedDifference(['SR_B6', 'SR_B5'])}
    for k, img in bands.items():
        pal, mn, mx = PAL[k]
        url = img.visualize(min=mn, max=mx, palette=pal).getThumbURL({'region': geom.bounds(), 'dimensions': 1400, 'format': 'png'})
        open(f'data/overlays/{k}_{year}.png', 'wb').write(requests.get(url).content)
    print('xong', year)


xong 2017
xong 2018
xong 2019
xong 2020
xong 2021
xong 2022
xong 2023
xong 2024
xong 2025
xong 2026


In [5]:
# Dán vào 1 cell của notebook (sau cell 1, đã có ee.Initialize(), maskL89Clouds, apply_scale_factors).
# Tạo dữ liệu TP.HCM SAU SÁP NHẬP (1/7/2025) = TP.HCM cũ + Bình Dương + Bà Rịa - Vũng Tàu.
import ee, json, os, requests, pandas as pd
os.makedirs('data/overlays', exist_ok=True)

# 1. Ranh giới mới = hợp 3 tỉnh/thành cũ (FAO GAUL). Cắt Côn Đảo (cách đất liền ~230 km) để raster không bị loãng.
adm = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.Or(
    ee.Filter.eq('ADM1_NAME', 'Ho Chi Minh City'),
    ee.Filter.stringContains('ADM1_NAME', 'Binh Duong'),
    ee.Filter.stringContains('ADM1_NAME', 'Vung Tau')))
names = adm.aggregate_array('ADM1_NAME').getInfo()
print('Đã chọn:', names)
assert len(names) == 3, 'Phải có đúng 3 đơn vị; kiểm tra lại tên trong GAUL'
roi = adm.geometry().intersection(ee.Geometry.Rectangle([106.0, 10.2, 107.7, 11.6]), 100)
json.dump(roi.getInfo(), open('data/hcm_boundary.geojson', 'w'), ensure_ascii=False)
b = roi.bounds().getInfo()['coordinates'][0]
json.dump({'west': min(p[0] for p in b), 'east': max(p[0] for p in b), 'south': min(p[1] for p in b), 'north': max(p[1] for p in b)},
          open('data/overlays/bounds.json', 'w'))
print('Diện tích ≈', round(roi.area(100).getInfo() / 1e6), 'km²')

# 2. Ảnh tổng hợp theo năm (Landsat 8 + 9)
def composite(year):
    col = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
           .filterBounds(roi).filterDate(f'{year}-01-01', f'{year}-12-31').filter(ee.Filter.lt('CLOUD_COVER', 20))
           .map(maskL89Clouds).map(apply_scale_factors).median().clip(roi))
    return ee.Image.cat([col.select('ST_B10').subtract(273.15).rename('LST'),
                         col.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI'),
                         col.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')])

# 3. CSV: 4 x 2.000 điểm CỐ ĐỊNH, lấy lại đúng các điểm đó mỗi năm (để có chuỗi thời gian từng điểm ảnh)
chunks = [ee.FeatureCollection.randomPoints(region=roi, points=2000, seed=s) for s in (1, 2, 3, 4)]
rows = []
for year in range(2017, 2027):
    img = composite(year)
    for pts in chunks:
        for f in img.sampleRegions(collection=pts, scale=30, geometries=True).getInfo()['features']:
            p = f['properties']
            rows.append({'LST': p.get('LST'), 'NDVI': p.get('NDVI'), 'NDBI': p.get('NDBI'), 'Year': year,
                         'longitude': f['geometry']['coordinates'][0], 'latitude': f['geometry']['coordinates'][1]})
    print('mẫu', year, len(rows))
df = pd.DataFrame(rows).dropna()
df.to_csv('data/HCMC_UHI_10Years_2017_2026.csv', index=False)
print(df.groupby('Year').size())

# 4. Ảnh raster PNG theo năm cho WebGIS (nếu báo quá giới hạn kích thước, giảm dimensions xuống 2000)
PAL = {'LST': (['blue', 'cyan', 'green', 'yellow', 'red'], 25, 45),
       'NDVI': (['red', 'yellow', 'lightgreen', 'green', 'darkgreen'], -0.2, 0.8),
       'NDBI': (['green', 'yellow', 'red'], -0.5, 0.5)}
for year in range(2017, 2027):
    img = composite(year)
    for k, (pal, mn, mx) in PAL.items():
        url = img.select(k).visualize(min=mn, max=mx, palette=pal).getThumbURL({'region': roi.bounds(), 'dimensions': 2000, 'format': 'png'})
        open(f'data/overlays/{k}_{year}.png', 'wb').write(requests.get(url).content)
    print('ảnh', year)


Đã chọn: ['Ba Ria-Vung Tau', 'Binh Duong', 'Ho Chi Minh City']
Diện tích ≈ 6602 km²
mẫu 2017 7949
mẫu 2018 15880
mẫu 2019 23818
mẫu 2020 31769
mẫu 2021 39725
mẫu 2022 47682
mẫu 2023 55640
mẫu 2024 63597
mẫu 2025 71548
mẫu 2026 79505
Year
2017    7949
2018    7931
2019    7938
2020    7951
2021    7956
2022    7957
2023    7958
2024    7957
2025    7951
2026    7957
dtype: int64
ảnh 2017
ảnh 2018
ảnh 2019
ảnh 2020
ảnh 2021
ảnh 2022
ảnh 2023
ảnh 2024
ảnh 2025
ảnh 2026


In [5]:
# Dán vào 1 cell của notebook (sau cell 1: đã có ee.Initialize() và apply_scale_factors).
# Xuất dữ liệu TP.HCM SAU SÁP NHẬP (1/7/2025) = TP.HCM cũ + Bình Dương + Bà Rịa - Vũng Tàu, gồm 4 chỉ số: LST, NDVI, NDBI, TVDI.
# Bản đã sửa lỗi "User memory limit exceeded": chọn trước band cần dùng, thêm tileScale, bỏ bestEffort ở bước tính biên.
import ee, json, os, requests, numpy as np, pandas as pd
os.makedirs('data/overlays', exist_ok=True)

# ---- Tham số chỉnh khi gặp lỗi bộ nhớ (tăng TILE_SCALE, tăng EDGE_SCALE) ----
BANDS = ['SR_B4', 'SR_B5', 'SR_B6', 'ST_B10', 'QA_PIXEL']  # chỉ giữ band cần dùng
EDGE_SCALE = 120   # độ phân giải (m) khi tính biên khô/ướt; nếu vẫn lỗi đổi thành 240
TILE_SCALE = 16    # chia nhỏ vùng tính toán cho reduceRegion/sampleRegions; nếu vẫn lỗi giữ 16 và tăng EDGE_SCALE

# 1. Ranh giới mới = hợp 3 tỉnh/thành cũ (FAO GAUL); cắt Côn Đảo (cách đất liền ~230 km)
adm = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.Or(
    ee.Filter.eq('ADM1_NAME', 'Ho Chi Minh City'),
    ee.Filter.stringContains('ADM1_NAME', 'Binh Duong'),
    ee.Filter.stringContains('ADM1_NAME', 'Vung Tau')))
names = adm.aggregate_array('ADM1_NAME').getInfo()
print('Đã chọn:', names)
assert len(names) == 3, 'Phải có đúng 3 đơn vị; kiểm tra lại tên trong GAUL'
roi = adm.geometry().intersection(ee.Geometry.Rectangle([106.0, 10.2, 107.7, 11.6]), 100)
json.dump(roi.getInfo(), open('data/hcm_boundary.geojson', 'w'), ensure_ascii=False)
b = roi.bounds().getInfo()['coordinates'][0]
json.dump({'west': min(p[0] for p in b), 'east': max(p[0] for p in b), 'south': min(p[1] for p in b), 'north': max(p[1] for p in b)},
          open('data/overlays/bounds.json', 'w'))
print('Diện tích ≈', round(roi.area(100).getInfo() / 1e6), 'km²')

# 2. Ảnh tổng hợp theo năm. Giảm pixel trống bằng 3 lớp:
#    (a) nới lọc cảnh (mây < 60%) + lọc mây theo pixel (mây, mây mở rộng, ti, bóng mây) => nhiều quan sát hơn cho median
#    (b) pixel vẫn trống => lấp bằng ảnh tổng hợp 3 năm quanh đó (năm-1 .. năm+1)
#    (c) lỗ nhỏ còn lại => lấp bằng median lân cận 7x7 pixel.   Băng QC ghi nguồn: 0 = quan sát thật, 1 = lấp từ năm lân cận, 2 = lấp lân cận.
def mask_clouds(img):
    return img.updateMask(img.select('QA_PIXEL').bitwiseAnd(0b11110).eq(0))

def landsat_median(start, end):
    # end là mốc loại trừ (filterDate), nên truyền 'YYYY+1-01-01' để lấy trọn năm
    col = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
           .filterBounds(roi).filterDate(start, end).filter(ee.Filter.lt('CLOUD_COVER', 60))
           .select(BANDS).map(mask_clouds).map(apply_scale_factors))
    m = col.median()
    lst = m.select('ST_B10').subtract(273.15).rename('LST')
    return ee.Image.cat([lst, m.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI'),
                         m.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')]).updateMask(lst.gt(15).And(lst.lt(65)))

# 3. TVDI = (LST - LSTmin) / (LSTmax - LSTmin); LSTmax = a_dry + b_dry*NDVI (biên khô), LSTmin = a_wet + b_wet*NDVI (biên ướt).
#    Biên được hồi quy tuyến tính theo từng năm: mỗi khoảng NDVI rộng 0,05 lấy phân vị 98% (khô) và 2% (ướt) của LST.
def fit_edges(groups, min_px=300):
    g = [d for d in groups if d['count'] >= min_px]
    x = np.array([d['bin'] + 0.025 for d in g])
    b_dry, a_dry = np.polyfit(x, [d['p98'] for d in g], 1)
    b_wet, a_wet = np.polyfit(x, [d['p2'] for d in g], 1)
    return a_dry, b_dry, a_wet, b_wet

EDGES, IMG = {}, {}
def composite(year):
    primary = landsat_median(f'{year}-01-01', f'{year+1}-01-01')
    nearby = landsat_median(f'{year-1}-01-01', f'{year+2}-01-01')
    f1 = ee.ImageCollection([nearby, primary]).mosaic()
    f2 = ee.ImageCollection([f1.focalMedian(3, 'square', 'pixels'), f1]).mosaic()
    qc = ee.Image(2).where(f1.select('LST').mask(), 1).where(primary.select('LST').mask(), 0).rename('QC')
    nd, lst = f2.select('NDVI'), f2.select('LST')
    bins = nd.multiply(20).floor().divide(20).rename('bin')
    red = (ee.Image.cat([lst, bins]).updateMask(nd.gt(0.05).And(nd.lt(0.85)))
           .reduceRegion(reducer=ee.Reducer.percentile([2, 98]).combine(ee.Reducer.count(), sharedInputs=True).group(1, 'bin'),
                         geometry=roi, scale=EDGE_SCALE, maxPixels=1e10, tileScale=TILE_SCALE).getInfo())
    a_dry, b_dry, a_wet, b_wet = EDGES[year] = fit_edges(red['groups'])
    lmax, lmin = nd.multiply(b_dry).add(a_dry), nd.multiply(b_wet).add(a_wet)
    tvdi = lst.subtract(lmin).divide(lmax.subtract(lmin).max(1)).clamp(0, 1).rename('TVDI')
    return f2.addBands(tvdi).addBands(qc).clip(roi)

for year in range(2017, 2027):
    IMG[year] = composite(year)
    ok = (IMG[year].select('LST').mask()
          .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=EDGE_SCALE, maxPixels=1e10, tileScale=TILE_SCALE)
          .getInfo()['LST'])
    print(year, '| pixel còn trống: %.2f%%' % (100 * (1 - ok)), '| biên khô/ướt:', [round(float(v), 2) for v in EDGES[year]])
json.dump({str(k): dict(zip(('a_dry', 'b_dry', 'a_wet', 'b_wet'), [float(x) for x in v])) for k, v in EDGES.items()},
          open('data/tvdi_edges.json', 'w'), indent=1)

# 4. CSV: 4 x 2.000 điểm CỐ ĐỊNH, lấy lại đúng các điểm đó mỗi năm (để có chuỗi thời gian từng điểm ảnh)
#    (mỗi lần getInfo tối đa 5.000 feature nên chia 4 nhóm 2.000 điểm)
chunks = [ee.FeatureCollection.randomPoints(region=roi, points=2000, seed=s) for s in (1, 2, 3, 4)]
rows = []
for year in range(2017, 2027):
    for pts in chunks:
        feats = IMG[year].sampleRegions(collection=pts, scale=30, geometries=True, tileScale=4).getInfo()['features']
        for f in feats:
            p = f['properties']
            rows.append({'LST': p.get('LST'), 'NDVI': p.get('NDVI'), 'NDBI': p.get('NDBI'), 'TVDI': p.get('TVDI'), 'QC': p.get('QC'),
                         'Year': year, 'longitude': f['geometry']['coordinates'][0], 'latitude': f['geometry']['coordinates'][1]})
    print('mẫu', year, len(rows))
df = pd.DataFrame(rows).dropna()
df.to_csv('data/HCMC_UHI_10Years_2017_2026.csv', index=False)
print(df.groupby('Year').size())

# 5. Ảnh raster PNG theo năm cho WebGIS: 4 lớp x 10 năm (nếu báo quá giới hạn kích thước, giảm dimensions xuống 2000)
PAL = {'LST': (['blue', 'cyan', 'green', 'yellow', 'red'], 25, 45),
       'NDVI': (['red', 'yellow', 'lightgreen', 'green', 'darkgreen'], -0.2, 0.8),
       'NDBI': (['green', 'yellow', 'red'], -0.5, 0.5),
       'TVDI': (['darkgreen', 'green', 'yellow', 'orange', 'red'], 0, 1)}
for year in range(2017, 2027):
    for k, (pal, mn, mx) in PAL.items():
        url = IMG[year].select(k).visualize(min=mn, max=mx, palette=pal).getThumbURL({'region': roi.bounds(), 'dimensions': 3000, 'format': 'png'})
        open(f'data/overlays/{k}_{year}.png', 'wb').write(requests.get(url).content)
    print('ảnh', year)


Đã chọn: ['Ba Ria-Vung Tau', 'Binh Duong', 'Ho Chi Minh City']
Diện tích ≈ 6602 km²
2017 | pixel còn trống: 0.26% | biên khô/ướt: [50.81, -15.73, 31.26, -0.53]
2018 | pixel còn trống: 0.26% | biên khô/ướt: [52.53, -18.86, 30.32, -0.26]
2019 | pixel còn trống: 0.26% | biên khô/ướt: [51.09, -16.36, 30.83, 1.57]
2020 | pixel còn trống: 0.25% | biên khô/ướt: [50.17, -15.65, 30.88, 0.18]
2021 | pixel còn trống: 0.25% | biên khô/ướt: [51.38, -16.84, 29.96, 1.98]
2022 | pixel còn trống: 0.25% | biên khô/ướt: [50.03, -16.61, 30.82, 0.36]
2023 | pixel còn trống: 0.25% | biên khô/ướt: [52.14, -17.32, 31.03, 1.25]
2024 | pixel còn trống: 0.25% | biên khô/ướt: [50.47, -16.3, 30.69, 1.0]
2025 | pixel còn trống: 0.25% | biên khô/ướt: [49.27, -17.6, 28.06, 0.18]
2026 | pixel còn trống: 0.26% | biên khô/ướt: [50.37, -16.17, 30.49, 0.5]
mẫu 2017 7991
mẫu 2018 15982
mẫu 2019 23973
mẫu 2020 31964
mẫu 2021 39955
mẫu 2022 47945
mẫu 2023 55936
mẫu 2024 63927
mẫu 2025 71918
mẫu 2026 79909
Year
2017    7991
2

EEException: Total request size (50760000 bytes) must be less than or equal to 50331648 bytes.

In [ ]:
# CHẠY TIẾP bước 5 (xuất PNG). Bước 1-4 đã xong: CSV đã có (7.991 điểm/năm), IMG và roi còn trong notebook.
# Dán vào 1 cell. Có thể ngắt/chạy lại thoải mái: ô đã tải được lưu ra đĩa và tự bỏ qua, ảnh đã xong không làm lại.
# Không đụng tới biến BANDS của notebook.
import os, io, json, time
import numpy as np
import requests
from PIL import Image, ImageDraw
from scipy import ndimage as ndi
from concurrent.futures import ThreadPoolExecutor

assert 'IMG' in globals() and 'roi' in globals(), 'Không thấy IMG/roi: cần chạy lại bước 2-3 của script chính trước'

DATA, OUT_DIR = 'data', 'data/overlays'
TILE_DIR = f'{OUT_DIR}/_tiles'
YEARS = list(range(2017, 2027))
W, N = 3000, 3            # chiều rộng ảnh ghép và lưới N x N (nếu ô nào hay lỗi thì tăng N lên 4 và xoá thư mục _tiles)
WORKERS = 4               # số ô tải cùng lúc; gặp lỗi quota/429 thì giảm còn 2
FILL_HOLES = True         # vá pixel trống trong ranh giới
SMOOTH_SIGMA, EDGE_ERODE = 4, 2
MAKE_EXCEL_GEOJSON = True # tạo Excel + GeoJSON từ CSV đã có (vài giây)

# ---------------- Excel + GeoJSON từ CSV đã có ----------------
if MAKE_EXCEL_GEOJSON:
    import pandas as pd
    CSV = f'{DATA}/HCMC_UHI_10Years_2017_2026.csv'
    df = pd.read_csv(CSV)
    IDX = ['LST', 'NDVI', 'NDBI', 'TVDI']
    summary = (df.groupby('Year').agg(LST_mean=('LST', 'mean'), NDVI_mean=('NDVI', 'mean'), NDBI_mean=('NDBI', 'mean'),
                                      TVDI_mean=('TVDI', 'mean'), n_diem=('LST', 'size')).round(3).reset_index())
    with pd.ExcelWriter(CSV.replace('.csv', '.xlsx'), engine='openpyxl') as xw:
        df.to_excel(xw, sheet_name='Data', index=False)
        summary.to_excel(xw, sheet_name='Thong_ke', index=False)
    feats = {}
    for r in df.itertuples(index=False):
        key = (round(r.longitude, 6), round(r.latitude, 6))
        if key not in feats:
            feats[key] = {'type': 'Feature', 'properties': {'id': len(feats)}, 'geometry': {'type': 'Point', 'coordinates': [key[0], key[1]]}}
        for k in IDX:
            feats[key]['properties'][f'{k}_{int(r.Year)}'] = round(float(getattr(r, k)), 3)
    with open(f'{DATA}/HCMC_UHI_points.geojson', 'w', encoding='utf-8') as fp:
        json.dump({'type': 'FeatureCollection', 'features': list(feats.values())}, fp, ensure_ascii=False, separators=(',', ':'))
    print(f'Excel + GeoJSON xong: {len(df)} dòng, {len(feats)} điểm (kỳ vọng ~8000 điểm)')

# ---------------- PNG theo ô ----------------
os.makedirs(TILE_DIR, exist_ok=True)
bb = json.load(open(f'{OUT_DIR}/bounds.json'))
west, east, south, north = bb['west'], bb['east'], bb['south'], bb['north']
H = round(W * (north - south) / (east - west))
xs = [round(i * W / N) for i in range(N + 1)]
ys = [round(i * H / N) for i in range(N + 1)]
dx, dy = (east - west) / W, (north - south) / H

PAL = {'LST': (['blue', 'cyan', 'green', 'yellow', 'red'], 25, 45),
       'NDVI': (['red', 'yellow', 'lightgreen', 'green', 'darkgreen'], -0.2, 0.8),
       'NDBI': (['green', 'yellow', 'red'], -0.5, 0.5),
       'TVDI': (['darkgreen', 'green', 'yellow', 'orange', 'red'], 0, 1)}

def _polys(g):
    if g['type'] == 'Polygon':
        return [g['coordinates']]
    if g['type'] == 'MultiPolygon':
        return g['coordinates']
    return [p for s in g.get('geometries', []) for p in _polys(s)]

def build_mask(w, h):
    m = Image.new('L', (w, h), 0)
    d = ImageDraw.Draw(m)
    px = lambda ring: [((x - west) / (east - west) * w, (north - y) / (north - south) * h) for x, y in ring]
    for poly in _polys(json.load(open(f'{DATA}/hcm_boundary.geojson'))):
        d.polygon(px(poly[0]), fill=255)
        for hole in poly[1:]:
            d.polygon(px(hole), fill=0)
    return ndi.binary_erosion(np.array(m) > 0, iterations=EDGE_ERODE)

INSIDE = build_mask(W, H) if FILL_HOLES else None

def fill_holes(img):
    a = np.array(img)
    valid = a[..., 3] > 0
    holes = INSIDE & ~valid
    if not holes.any():
        return img, 0.0
    iy, ix = ndi.distance_transform_edt(~valid, return_distances=False, return_indices=True)
    filled = a[iy, ix]
    rgb = (np.stack([ndi.gaussian_filter(filled[..., c].astype(np.float32), SMOOTH_SIGMA) for c in range(3)], -1)
           if SMOOTH_SIGMA > 0 else filled[..., :3].astype(np.float32))
    a[holes, :3] = np.clip(rgb[holes], 0, 255).astype(np.uint8)
    a[holes, 3] = 255
    return Image.fromarray(a, 'RGBA'), 100 * holes.sum() / INSIDE.sum()

def tile_url(vis, c0, c1, r0, r1):
    rect = ee.Geometry.Rectangle([west + c0 * dx, north - r1 * dy, west + c1 * dx, north - r0 * dy], 'EPSG:4326', False)
    return vis.getThumbURL({'region': rect, 'dimensions': f'{c1 - c0}x{r1 - r0}', 'format': 'png', 'crs': 'EPSG:4326'})

def download(url):
    try:
        r = requests.get(url, timeout=900)
        return (r.status_code, r.content if r.status_code == 200 else r.text[:500])
    except Exception as e:
        return (0, str(e))

def tile_path(k, year, i, j):
    return f'{TILE_DIR}/{k}_{year}_{i}_{j}.png'

def save_atomic(img, path):
    tmp = path + '.tmp'
    img.save(tmp, format='PNG')
    os.replace(tmp, path)

FAILED = []
t_all = time.time()
for year in YEARS:
    t0 = time.time()
    # 1) ảnh đã có: chỉ vá lỗ (nếu bật)
    todo = []
    for k in PAL:
        p = f'{OUT_DIR}/{k}_{year}.png'
        if os.path.exists(p):
            if FILL_HOLES:
                im = Image.open(p).convert('RGBA')
                if im.size == (W, H):
                    im, pct = fill_holes(im)
                    if pct:
                        save_atomic(im, p)
                        print(f'  {k}_{year}: vá {pct:.2f}% (ảnh cũ)')
        else:
            todo.append(k)
    if not todo:
        continue

    # 2) liệt kê ô còn thiếu, tạo URL ở luồng chính, tải song song, lưu từng ô ra đĩa ngay
    jobs = {}
    for k in todo:
        pal, mn, mx = PAL[k]
        vis = None
        for i in range(N):
            for j in range(N):
                if os.path.exists(tile_path(k, year, i, j)):
                    continue
                if vis is None:
                    vis = IMG[year].select(k).visualize(min=mn, max=mx, palette=pal)
                jobs[(k, i, j)] = (vis, tile_url(vis, xs[j], xs[j + 1], ys[i], ys[i + 1]))
    keys = list(jobs)
    if keys:
        with ThreadPoolExecutor(WORKERS) as ex:
            results = dict(zip(keys, ex.map(download, [jobs[q][1] for q in keys])))
        for q in keys:
            k, i, j = q
            for attempt in range(3):
                if results[q][0] == 200:
                    break
                print(f'  ô lỗi {k}_{year} ({i},{j}) lần {attempt + 1}: HTTP {results[q][0]} {str(results[q][1])[:200]}')
                time.sleep(5)
                results[q] = download(tile_url(jobs[q][0], xs[j], xs[j + 1], ys[i], ys[i + 1]))
            if results[q][0] == 200:
                with open(tile_path(k, year, i, j), 'wb') as fp:
                    fp.write(results[q][1])
            else:
                FAILED.append(q + (year,))

    # 3) ghép ảnh nào đã đủ 9 ô
    for k in todo:
        paths = {(i, j): tile_path(k, year, i, j) for i in range(N) for j in range(N)}
        if not all(os.path.exists(p) for p in paths.values()):
            continue
        canvas = Image.new('RGBA', (W, H), (0, 0, 0, 0))
        for (i, j), p in paths.items():
            c0, c1, r0, r1 = xs[j], xs[j + 1], ys[i], ys[i + 1]
            t = Image.open(p).convert('RGBA')
            canvas.paste(t if t.size == (c1 - c0, r1 - r0) else t.resize((c1 - c0, r1 - r0)), (c0, r0))
        if FILL_HOLES:
            canvas, pct = fill_holes(canvas)
            print(f'  {k}_{year}: vá {pct:.2f}%')
        save_atomic(canvas, f'{OUT_DIR}/{k}_{year}.png')
        for p in paths.values():
            os.remove(p)       # ảnh đã xong, xoá ô tạm
    print(f'ảnh {year} xong ({time.time() - t0:.0f}s, tổng {time.time() - t_all:.0f}s)')

if FAILED:
    print('\nCòn ô chưa tải được (chạy lại cell này sẽ tải tiếp đúng các ô này):')
    for k, i, j, y in FAILED:
        print(f'  {k}_{y} ô ({i},{j})')
else:
    print('\nXong toàn bộ PNG.')


Excel + GeoJSON xong: 79909 dòng, 7986 điểm (kỳ vọng ~8000 điểm)
  LST_2017: vá 0.08%
  NDVI_2017: vá 0.08%
  NDBI_2017: vá 0.08%
  TVDI_2017: vá 0.08%
ảnh 2017 xong (474s, tổng 474s)


FileNotFoundError: [Errno 2] No such file or directory: 'data/overlays/_tiles/LST_2018_0_0.png'

In [20]:
# CHẠY TIẾP bước 5 (xuất PNG). Bước 1-4 đã xong: CSV đã có (7.991 điểm/năm), IMG và roi còn trong notebook.
# Dán vào 1 cell. Có thể ngắt/chạy lại thoải mái: ô đã tải được lưu ra đĩa và tự bỏ qua, ảnh đã xong không làm lại.
# Không đụng tới biến BANDS của notebook.
import os, io, json, time
import numpy as np
import requests
from PIL import Image, ImageDraw
from scipy import ndimage as ndi
from concurrent.futures import ThreadPoolExecutor

assert 'IMG' in globals() and 'roi' in globals(), 'Không thấy IMG/roi: cần chạy lại bước 2-3 của script chính trước'

DATA, OUT_DIR = 'data', 'data/overlays'
TILE_DIR = f'{OUT_DIR}/_tiles'
YEARS = list(range(2017, 2027))
W, N = 3000, 3            # chiều rộng ảnh ghép và lưới N x N (nếu ô nào hay lỗi thì tăng N lên 4 và xoá thư mục _tiles)
WORKERS = 4               # số ô tải cùng lúc; gặp lỗi quota/429 thì giảm còn 2
FILL_HOLES = True         # vá pixel trống trong ranh giới
SMOOTH_SIGMA, EDGE_ERODE = 4, 2
MAKE_EXCEL_GEOJSON = True # tạo Excel + GeoJSON từ CSV đã có (vài giây)

# ---------------- Excel + GeoJSON từ CSV đã có ----------------
if MAKE_EXCEL_GEOJSON:
    import pandas as pd
    CSV = f'{DATA}/HCMC_UHI_10Years_2017_2026.csv'
    df = pd.read_csv(CSV)
    IDX = ['LST', 'NDVI', 'NDBI', 'TVDI']
    summary = (df.groupby('Year').agg(LST_mean=('LST', 'mean'), NDVI_mean=('NDVI', 'mean'), NDBI_mean=('NDBI', 'mean'),
                                      TVDI_mean=('TVDI', 'mean'), n_diem=('LST', 'size')).round(3).reset_index())
    with pd.ExcelWriter(CSV.replace('.csv', '.xlsx'), engine='openpyxl') as xw:
        df.to_excel(xw, sheet_name='Data', index=False)
        summary.to_excel(xw, sheet_name='Thong_ke', index=False)
    feats = {}
    for r in df.itertuples(index=False):
        key = (round(r.longitude, 6), round(r.latitude, 6))
        if key not in feats:
            feats[key] = {'type': 'Feature', 'properties': {'id': len(feats)}, 'geometry': {'type': 'Point', 'coordinates': [key[0], key[1]]}}
        for k in IDX:
            feats[key]['properties'][f'{k}_{int(r.Year)}'] = round(float(getattr(r, k)), 3)
    with open(f'{DATA}/HCMC_UHI_points.geojson', 'w', encoding='utf-8') as fp:
        json.dump({'type': 'FeatureCollection', 'features': list(feats.values())}, fp, ensure_ascii=False, separators=(',', ':'))
    print(f'Excel + GeoJSON xong: {len(df)} dòng, {len(feats)} điểm (kỳ vọng ~8000 điểm)')

# ---------------- PNG theo ô ----------------
os.makedirs(TILE_DIR, exist_ok=True)
bb = json.load(open(f'{OUT_DIR}/bounds.json'))
west, east, south, north = bb['west'], bb['east'], bb['south'], bb['north']
H = round(W * (north - south) / (east - west))
xs = [round(i * W / N) for i in range(N + 1)]
ys = [round(i * H / N) for i in range(N + 1)]
dx, dy = (east - west) / W, (north - south) / H

PAL = {'LST': (['blue', 'cyan', 'green', 'yellow', 'red'], 25, 45),
       'NDVI': (['red', 'yellow', 'lightgreen', 'green', 'darkgreen'], -0.2, 0.8),
       'NDBI': (['green', 'yellow', 'red'], -0.5, 0.5),
       'TVDI': (['darkgreen', 'green', 'yellow', 'orange', 'red'], 0, 1)}

def _polys(g):
    if g['type'] == 'Polygon':
        return [g['coordinates']]
    if g['type'] == 'MultiPolygon':
        return g['coordinates']
    return [p for s in g.get('geometries', []) for p in _polys(s)]

def build_mask(w, h):
    m = Image.new('L', (w, h), 0)
    d = ImageDraw.Draw(m)
    px = lambda ring: [((x - west) / (east - west) * w, (north - y) / (north - south) * h) for x, y in ring]
    for poly in _polys(json.load(open(f'{DATA}/hcm_boundary.geojson'))):
        d.polygon(px(poly[0]), fill=255)
        for hole in poly[1:]:
            d.polygon(px(hole), fill=0)
    return ndi.binary_erosion(np.array(m) > 0, iterations=EDGE_ERODE)

INSIDE = build_mask(W, H) if FILL_HOLES else None

def fill_holes(img):
    a = np.array(img)
    valid = a[..., 3] > 0
    holes = INSIDE & ~valid
    if not holes.any():
        return img, 0.0
    iy, ix = ndi.distance_transform_edt(~valid, return_distances=False, return_indices=True)
    filled = a[iy, ix]
    rgb = (np.stack([ndi.gaussian_filter(filled[..., c].astype(np.float32), SMOOTH_SIGMA) for c in range(3)], -1)
           if SMOOTH_SIGMA > 0 else filled[..., :3].astype(np.float32))
    a[holes, :3] = np.clip(rgb[holes], 0, 255).astype(np.uint8)
    a[holes, 3] = 255
    return Image.fromarray(a, 'RGBA'), 100 * holes.sum() / INSIDE.sum()

def tile_url(vis, c0, c1, r0, r1):
    rect = ee.Geometry.Rectangle([west + c0 * dx, north - r1 * dy, west + c1 * dx, north - r0 * dy], 'EPSG:4326', False)
    return vis.getThumbURL({'region': rect, 'dimensions': f'{c1 - c0}x{r1 - r0}', 'format': 'png', 'crs': 'EPSG:4326'})

def download(url):
    try:
        r = requests.get(url, timeout=900)
        return (r.status_code, r.content if r.status_code == 200 else r.text[:500])
    except Exception as e:
        return (0, str(e))

def tile_path(k, year, i, j):
    return f'{TILE_DIR}/{k}_{year}_{i}_{j}.png'

def save_atomic(img, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    img.save(tmp, format='PNG')
    os.replace(tmp, path)

FAILED = []
t_all = time.time()
for year in YEARS:
    t0 = time.time()
    # 1) ảnh đã có: chỉ vá lỗ (nếu bật)
    todo = []
    for k in PAL:
        p = f'{OUT_DIR}/{k}_{year}.png'
        if os.path.exists(p):
            if FILL_HOLES:
                im = Image.open(p).convert('RGBA')
                if im.size == (W, H):
                    im, pct = fill_holes(im)
                    if pct:
                        save_atomic(im, p)
                        print(f'  {k}_{year}: vá {pct:.2f}% (ảnh cũ)')
        else:
            todo.append(k)
    if not todo:
        continue

    # 2) liệt kê ô còn thiếu, tạo URL ở luồng chính, tải song song, lưu từng ô ra đĩa ngay
    jobs = {}
    for k in todo:
        pal, mn, mx = PAL[k]
        vis = None
        for i in range(N):
            for j in range(N):
                if os.path.exists(tile_path(k, year, i, j)):
                    continue
                if vis is None:
                    vis = IMG[year].select(k).visualize(min=mn, max=mx, palette=pal)
                jobs[(k, i, j)] = (vis, tile_url(vis, xs[j], xs[j + 1], ys[i], ys[i + 1]))
    keys = list(jobs)
    if keys:
        with ThreadPoolExecutor(WORKERS) as ex:
            results = dict(zip(keys, ex.map(download, [jobs[q][1] for q in keys])))
        for q in keys:
            k, i, j = q
            for attempt in range(3):
                if results[q][0] == 200:
                    break
                print(f'  ô lỗi {k}_{year} ({i},{j}) lần {attempt + 1}: HTTP {results[q][0]} {str(results[q][1])[:200]}')
                time.sleep(5)
                results[q] = download(tile_url(jobs[q][0], xs[j], xs[j + 1], ys[i], ys[i + 1]))
            if results[q][0] == 200:
                os.makedirs(TILE_DIR, exist_ok=True)   # tạo lại nếu thư mục bị xoá giữa chừng
                with open(tile_path(k, year, i, j), 'wb') as fp:
                    fp.write(results[q][1])
            else:
                FAILED.append(q + (year,))

    # 3) ghép ảnh nào đã đủ 9 ô
    for k in todo:
        paths = {(i, j): tile_path(k, year, i, j) for i in range(N) for j in range(N)}
        if not all(os.path.exists(p) for p in paths.values()):
            continue
        canvas = Image.new('RGBA', (W, H), (0, 0, 0, 0))
        for (i, j), p in paths.items():
            c0, c1, r0, r1 = xs[j], xs[j + 1], ys[i], ys[i + 1]
            t = Image.open(p).convert('RGBA')
            canvas.paste(t if t.size == (c1 - c0, r1 - r0) else t.resize((c1 - c0, r1 - r0)), (c0, r0))
        if FILL_HOLES:
            canvas, pct = fill_holes(canvas)
            print(f'  {k}_{year}: vá {pct:.2f}%')
        save_atomic(canvas, f'{OUT_DIR}/{k}_{year}.png')
        for p in paths.values():
            os.remove(p)       # ảnh đã xong, xoá ô tạm
    print(f'ảnh {year} xong ({time.time() - t0:.0f}s, tổng {time.time() - t_all:.0f}s)')

if FAILED:
    print('\nCòn ô chưa tải được (chạy lại cell này sẽ tải tiếp đúng các ô này):')
    for k, i, j, y in FAILED:
        print(f'  {k}_{y} ô ({i},{j})')
else:
    print('\nXong toàn bộ PNG.')


Excel + GeoJSON xong: 79909 dòng, 7986 điểm (kỳ vọng ~8000 điểm)
  LST_2018: vá 0.08%
  NDVI_2018: vá 0.08%
  NDBI_2018: vá 0.08%
  TVDI_2018: vá 0.08%
ảnh 2018 xong (478s, tổng 479s)
  LST_2019: vá 0.08%
  NDVI_2019: vá 0.08%
  NDBI_2019: vá 0.08%
  TVDI_2019: vá 0.08%
ảnh 2019 xong (553s, tổng 1032s)
  LST_2020: vá 0.08%
  NDVI_2020: vá 0.08%
  NDBI_2020: vá 0.08%
  TVDI_2020: vá 0.08%
ảnh 2020 xong (589s, tổng 1620s)
  ô lỗi NDVI_2021 (1,1) lần 1: HTTP 400 {
  "error": {
    "code": 400,
    "message": "User memory limit exceeded.",
    "status": "INVALID_ARGUMENT"
  }
}

  ô lỗi NDVI_2021 (1,1) lần 2: HTTP 400 {
  "error": {
    "code": 400,
    "message": "User memory limit exceeded.",
    "status": "INVALID_ARGUMENT"
  }
}

  ô lỗi NDVI_2021 (1,1) lần 3: HTTP 400 {
  "error": {
    "code": 400,
    "message": "User memory limit exceeded.",
    "status": "INVALID_ARGUMENT"
  }
}

  ô lỗi NDBI_2021 (1,1) lần 1: HTTP 400 {
  "error": {
    "code": 400,
    "message": "User memory li

KeyboardInterrupt: 

In [21]:
# CHẠY TIẾP bước 5 (xuất PNG). Bước 1-4 đã xong: CSV đã có (7.991 điểm/năm), IMG và roi còn trong notebook.
# Dán vào 1 cell. Có thể ngắt/chạy lại thoải mái: ô đã tải được lưu ra đĩa và tự bỏ qua, ảnh đã xong không làm lại.
# Không đụng tới biến BANDS của notebook.
import os, io, json, time
import numpy as np
import requests
from PIL import Image, ImageDraw
from scipy import ndimage as ndi
from concurrent.futures import ThreadPoolExecutor

assert 'IMG' in globals() and 'roi' in globals(), 'Không thấy IMG/roi: cần chạy lại bước 2-3 của script chính trước'

DATA, OUT_DIR = 'data', 'data/overlays'
TILE_DIR = f'{OUT_DIR}/_tiles'
YEARS = list(range(2017, 2027))
W, N = 3000, 3            # chiều rộng ảnh ghép và lưới N x N (nếu ô nào hay lỗi thì tăng N lên 4 và xoá thư mục _tiles)
WORKERS = 4               # số ô tải cùng lúc; gặp lỗi quota/429 thì giảm còn 2
FILL_HOLES = True         # vá pixel trống trong ranh giới
SMOOTH_SIGMA, EDGE_ERODE = 4, 2
MAKE_EXCEL_GEOJSON = True # tạo Excel + GeoJSON từ CSV đã có (vài giây)

# ---------------- Excel + GeoJSON từ CSV đã có ----------------
if MAKE_EXCEL_GEOJSON:
    import pandas as pd
    CSV = f'{DATA}/HCMC_UHI_10Years_2017_2026.csv'
    df = pd.read_csv(CSV)
    IDX = ['LST', 'NDVI', 'NDBI', 'TVDI']
    summary = (df.groupby('Year').agg(LST_mean=('LST', 'mean'), NDVI_mean=('NDVI', 'mean'), NDBI_mean=('NDBI', 'mean'),
                                      TVDI_mean=('TVDI', 'mean'), n_diem=('LST', 'size')).round(3).reset_index())
    xlsx = CSV.replace('.csv', '.xlsx')
    try:
        open(xlsx, 'ab').close()      # thử mở để kiểm tra file có đang bị khoá (đang mở trong Excel) không
    except PermissionError:
        xlsx = xlsx.replace('.xlsx', f'_{time.strftime("%H%M%S")}.xlsx')
        print('File Excel đang được mở ở chương trình khác, ghi sang:', xlsx)
    with pd.ExcelWriter(xlsx, engine='openpyxl') as xw:
        df.to_excel(xw, sheet_name='Data', index=False)
        summary.to_excel(xw, sheet_name='Thong_ke', index=False)
    feats = {}
    for r in df.itertuples(index=False):
        key = (round(r.longitude, 6), round(r.latitude, 6))
        if key not in feats:
            feats[key] = {'type': 'Feature', 'properties': {'id': len(feats)}, 'geometry': {'type': 'Point', 'coordinates': [key[0], key[1]]}}
        for k in IDX:
            feats[key]['properties'][f'{k}_{int(r.Year)}'] = round(float(getattr(r, k)), 3)
    with open(f'{DATA}/HCMC_UHI_points.geojson', 'w', encoding='utf-8') as fp:
        json.dump({'type': 'FeatureCollection', 'features': list(feats.values())}, fp, ensure_ascii=False, separators=(',', ':'))
    print(f'Excel + GeoJSON xong: {len(df)} dòng, {len(feats)} điểm (kỳ vọng ~8000 điểm)')

# ---------------- PNG theo ô ----------------
os.makedirs(TILE_DIR, exist_ok=True)
bb = json.load(open(f'{OUT_DIR}/bounds.json'))
west, east, south, north = bb['west'], bb['east'], bb['south'], bb['north']
H = round(W * (north - south) / (east - west))
xs = [round(i * W / N) for i in range(N + 1)]
ys = [round(i * H / N) for i in range(N + 1)]
dx, dy = (east - west) / W, (north - south) / H

PAL = {'LST': (['blue', 'cyan', 'green', 'yellow', 'red'], 25, 45),
       'NDVI': (['red', 'yellow', 'lightgreen', 'green', 'darkgreen'], -0.2, 0.8),
       'NDBI': (['green', 'yellow', 'red'], -0.5, 0.5),
       'TVDI': (['darkgreen', 'green', 'yellow', 'orange', 'red'], 0, 1)}

def _polys(g):
    if g['type'] == 'Polygon':
        return [g['coordinates']]
    if g['type'] == 'MultiPolygon':
        return g['coordinates']
    return [p for s in g.get('geometries', []) for p in _polys(s)]

def build_mask(w, h):
    m = Image.new('L', (w, h), 0)
    d = ImageDraw.Draw(m)
    px = lambda ring: [((x - west) / (east - west) * w, (north - y) / (north - south) * h) for x, y in ring]
    for poly in _polys(json.load(open(f'{DATA}/hcm_boundary.geojson'))):
        d.polygon(px(poly[0]), fill=255)
        for hole in poly[1:]:
            d.polygon(px(hole), fill=0)
    return ndi.binary_erosion(np.array(m) > 0, iterations=EDGE_ERODE)

INSIDE = build_mask(W, H) if FILL_HOLES else None

def fill_holes(img):
    a = np.array(img)
    valid = a[..., 3] > 0
    holes = INSIDE & ~valid
    if not holes.any():
        return img, 0.0
    iy, ix = ndi.distance_transform_edt(~valid, return_distances=False, return_indices=True)
    filled = a[iy, ix]
    rgb = (np.stack([ndi.gaussian_filter(filled[..., c].astype(np.float32), SMOOTH_SIGMA) for c in range(3)], -1)
           if SMOOTH_SIGMA > 0 else filled[..., :3].astype(np.float32))
    a[holes, :3] = np.clip(rgb[holes], 0, 255).astype(np.uint8)
    a[holes, 3] = 255
    return Image.fromarray(a, 'RGBA'), 100 * holes.sum() / INSIDE.sum()

def tile_url(vis, c0, c1, r0, r1):
    rect = ee.Geometry.Rectangle([west + c0 * dx, north - r1 * dy, west + c1 * dx, north - r0 * dy], 'EPSG:4326', False)
    return vis.getThumbURL({'region': rect, 'dimensions': f'{c1 - c0}x{r1 - r0}', 'format': 'png', 'crs': 'EPSG:4326'})

def download(url):
    try:
        r = requests.get(url, timeout=900)
        return (r.status_code, r.content if r.status_code == 200 else r.text[:500])
    except Exception as e:
        return (0, str(e))

def tile_path(k, year, i, j):
    tag = '' if N == 3 else f'_n{N}'     # đổi N thì tên ô khác đi, không lẫn với ô cũ
    return f'{TILE_DIR}/{k}_{year}{tag}_{i}_{j}.png'

def fetch_split(vis, c0, c1, r0, r1, depth=0):
    """Tải 1 ô; nếu lỗi thì chia làm 4 ô con (tối đa 2 cấp) rồi ghép lại. Dùng cho ô quá nặng (lỗi bộ nhớ)."""
    code, body = download(tile_url(vis, c0, c1, r0, r1))
    if code == 200:
        t = Image.open(io.BytesIO(body)).convert('RGBA')
        return t if t.size == (c1 - c0, r1 - r0) else t.resize((c1 - c0, r1 - r0))
    if depth >= 2:
        raise RuntimeError(' '.join(str(body).split())[:150])
    cm, rm = (c0 + c1) // 2, (r0 + r1) // 2
    out = Image.new('RGBA', (c1 - c0, r1 - r0), (0, 0, 0, 0))
    for a, b in ((c0, cm), (cm, c1)):
        for c, d in ((r0, rm), (rm, r1)):
            out.paste(fetch_split(vis, a, b, c, d, depth + 1), (a - c0, c - r0))
    return out

def save_atomic(img, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    img.save(tmp, format='PNG')
    os.replace(tmp, path)

FAILED = []
t_all = time.time()
for year in YEARS:
    t0 = time.time()
    # 1) ảnh đã có: chỉ vá lỗ (nếu bật)
    todo = []
    for k in PAL:
        p = f'{OUT_DIR}/{k}_{year}.png'
        if os.path.exists(p):
            if FILL_HOLES:
                im = Image.open(p).convert('RGBA')
                if im.size == (W, H):
                    im, pct = fill_holes(im)
                    if pct:
                        save_atomic(im, p)
                        print(f'  {k}_{year}: vá {pct:.2f}% (ảnh cũ)')
        else:
            todo.append(k)
    if not todo:
        continue

    # 2) liệt kê ô còn thiếu, tạo URL ở luồng chính, tải song song, lưu từng ô ra đĩa ngay
    jobs = {}
    for k in todo:
        pal, mn, mx = PAL[k]
        vis = None
        for i in range(N):
            for j in range(N):
                if os.path.exists(tile_path(k, year, i, j)):
                    continue
                if vis is None:
                    vis = IMG[year].select(k).visualize(min=mn, max=mx, palette=pal)
                jobs[(k, i, j)] = (vis, tile_url(vis, xs[j], xs[j + 1], ys[i], ys[i + 1]))
    keys = list(jobs)
    if keys:
        with ThreadPoolExecutor(WORKERS) as ex:
            results = dict(zip(keys, ex.map(download, [jobs[q][1] for q in keys])))
        for q in keys:
            k, i, j = q
            for attempt in range(3):
                code, body = results[q]
                if code == 200:
                    break
                msg = ' '.join(str(body).split())[:150]
                if 'memory' in msg.lower():     # lỗi bộ nhớ: thử lại cùng kích thước là vô ích -> chia nhỏ ô
                    print(f'  ô {k}_{year} ({i},{j}) quá nặng (bộ nhớ), chia nhỏ ô...')
                    break
                print(f'  ô lỗi {k}_{year} ({i},{j}) lần {attempt + 1}: HTTP {code} {msg}')
                time.sleep(5)
                results[q] = download(tile_url(jobs[q][0], xs[j], xs[j + 1], ys[i], ys[i + 1]))
            os.makedirs(TILE_DIR, exist_ok=True)   # tạo lại nếu thư mục bị xoá giữa chừng
            if results[q][0] == 200:
                with open(tile_path(k, year, i, j), 'wb') as fp:
                    fp.write(results[q][1])
            else:
                try:
                    fetch_split(jobs[q][0], xs[j], xs[j + 1], ys[i], ys[i + 1]).save(tile_path(k, year, i, j), format='PNG')
                except Exception as e:
                    print(f'  ô {k}_{year} ({i},{j}) vẫn lỗi sau khi chia nhỏ: {e}')
                    FAILED.append(q + (year,))

    # 3) ghép ảnh nào đã đủ 9 ô
    for k in todo:
        paths = {(i, j): tile_path(k, year, i, j) for i in range(N) for j in range(N)}
        if not all(os.path.exists(p) for p in paths.values()):
            continue
        canvas = Image.new('RGBA', (W, H), (0, 0, 0, 0))
        for (i, j), p in paths.items():
            c0, c1, r0, r1 = xs[j], xs[j + 1], ys[i], ys[i + 1]
            t = Image.open(p).convert('RGBA')
            canvas.paste(t if t.size == (c1 - c0, r1 - r0) else t.resize((c1 - c0, r1 - r0)), (c0, r0))
        if FILL_HOLES:
            canvas, pct = fill_holes(canvas)
            print(f'  {k}_{year}: vá {pct:.2f}%')
        save_atomic(canvas, f'{OUT_DIR}/{k}_{year}.png')
        for p in paths.values():
            os.remove(p)       # ảnh đã xong, xoá ô tạm
    print(f'ảnh {year} xong ({time.time() - t0:.0f}s, tổng {time.time() - t_all:.0f}s)')

if FAILED:
    print('\nCòn ô chưa tải được (chạy lại cell này sẽ tải tiếp đúng các ô này):')
    for k, i, j, y in FAILED:
        print(f'  {k}_{y} ô ({i},{j})')
else:
    print('\nXong toàn bộ PNG.')


Excel + GeoJSON xong: 79909 dòng, 7986 điểm (kỳ vọng ~8000 điểm)
  ô NDVI_2021 (1,1) quá nặng (bộ nhớ), chia nhỏ ô...
  ô NDBI_2021 (1,1) quá nặng (bộ nhớ), chia nhỏ ô...
  ô TVDI_2021 (1,1) quá nặng (bộ nhớ), chia nhỏ ô...
  NDVI_2021: vá 0.08%
  NDBI_2021: vá 0.08%
  TVDI_2021: vá 0.08%
ảnh 2021 xong (281s, tổng 284s)
  ô NDVI_2022 (0,1) quá nặng (bộ nhớ), chia nhỏ ô...
  ô NDVI_2022 (1,1) quá nặng (bộ nhớ), chia nhỏ ô...
  ô NDVI_2022 (1,2) quá nặng (bộ nhớ), chia nhỏ ô...
  ô NDVI_2022 (2,1) quá nặng (bộ nhớ), chia nhỏ ô...
  ô NDBI_2022 (1,1) quá nặng (bộ nhớ), chia nhỏ ô...
  ô NDBI_2022 (1,2) quá nặng (bộ nhớ), chia nhỏ ô...
  ô NDBI_2022 (2,1) quá nặng (bộ nhớ), chia nhỏ ô...
  ô TVDI_2022 (0,1) quá nặng (bộ nhớ), chia nhỏ ô...
  ô TVDI_2022 (1,1) quá nặng (bộ nhớ), chia nhỏ ô...
  ô TVDI_2022 (1,2) quá nặng (bộ nhớ), chia nhỏ ô...
  ô TVDI_2022 (2,1) quá nặng (bộ nhớ), chia nhỏ ô...
  LST_2022: vá 0.07%
  NDVI_2022: vá 0.07%
  NDBI_2022: vá 0.07%
  TVDI_2022: vá 0.07%
ảnh 2022